# Lab 10: Fundamentals of Generative AI & Prompt Engineering

---

## Aim
To understand the fundamentals of Generative AI and experiment with prompt-based tools using the Hugging Face Transformers library and the OpenAI-compatible API interface.

## Theory

### What is Generative AI?
Generative AI refers to models that can **generate new content** (text, images, code, audio) based on learned patterns from training data. Unlike traditional ML models that classify or predict, generative models *create*.

### Key Concepts:
| Concept | Description |
|---|---|
| **LLM** | Large Language Model — a neural network trained on massive text corpora |
| **Prompt** | The input instruction given to a generative model |
| **Token** | The smallest unit of text a model processes (word-piece) |
| **Temperature** | Controls randomness in output (0 = deterministic, 1 = creative) |
| **Context Window** | Maximum number of tokens a model can consider at once |
| **Zero-shot** | Model answers without any examples in the prompt |
| **Few-shot** | Model is given a few examples before the actual query |

### How do LLMs Generate Text?
LLMs use the **next-token prediction** paradigm. Given a sequence of tokens, the model predicts the probability distribution over the vocabulary for the next token, then samples from it.

```
Input:  "The capital of France is"
Model:  P(Paris) = 0.92, P(Lyon) = 0.03, ...
Output: "Paris"
```

---

##  Part 1: Environment Setup

We use **Hugging Face Transformers** — an open-source library that provides thousands of pretrained models. This avoids needing paid API keys while still working with real LLMs.

In [1]:
# Install required libraries
# Run this cell once; restart kernel if needed
!pip install transformers torch sentencepiece accelerate -q

In [2]:
# Core imports
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded successfully")

✅ Libraries loaded successfully


---
##  Part 2: Exploring Tokenization

Before a model reads text, it converts words into **tokens** using a tokenizer.  
Understanding tokenization is fundamental — it tells us *how the model sees language*.

In [3]:
from transformers import AutoTokenizer

# Load the GPT-2 tokenizer (lightweight, no GPU needed)
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# --- Experiment 1: See how text becomes tokens ---
sample_text = "Generative AI is transforming how we interact with computers."

# Encode text → token IDs
token_ids = tokenizer.encode(sample_text)

# Decode IDs back → readable tokens
tokens = [tokenizer.decode([tid]) for tid in token_ids]

print(f"Original Text : {sample_text}")
print(f"\nToken Count   : {len(token_ids)}")
print(f"\nToken IDs     : {token_ids}")
print(f"\nTokens        : {tokens}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Original Text : Generative AI is transforming how we interact with computers.

Token Count   : 11

Token IDs     : [8645, 876, 9552, 318, 25449, 703, 356, 9427, 351, 9061, 13]

Tokens        : ['Gener', 'ative', ' AI', ' is', ' transforming', ' how', ' we', ' interact', ' with', ' computers', '.']


In [4]:
# --- Experiment 2: Compare token counts across sentence types ---
# This shows how token count varies with content complexity

sentences = [
    "Hello world.",
    "The quick brown fox jumps over the lazy dog.",
    "Supercalifragilistic expialidocious is a fun word.",
    "मशीन लर्निंग एक रोचक विषय है।"   # Hindi — shows subword tokenization
]

print(f"{'Sentence':<50} | Tokens")
print("-" * 65)
for s in sentences:
    count = len(tokenizer.encode(s))
    print(f"{s:<50} | {count}")

print("\n📌 Observation: Non-English text uses more tokens due to subword splitting.")

Sentence                                           | Tokens
-----------------------------------------------------------------
Hello world.                                       | 3
The quick brown fox jumps over the lazy dog.       | 10
Supercalifragilistic expialidocious is a fun word. | 15
मशीन लर्निंग एक रोचक विषय है।                      | 48

📌 Observation: Non-English text uses more tokens due to subword splitting.


** Observation:** Tokenizers split unknown or rare words into *subword* units. This is why languages with complex morphology (like Hindi) use more tokens per word than English.

---
##  Part 3: Text Generation with a Pretrained LLM

We use GPT-2 (124M parameters) — a classic open-source autoregressive language model.  
The `pipeline` abstraction makes it easy to load and run models in just a few lines.

In [5]:
# Load the text-generation pipeline with GPT-2
# GPT-2 is small enough to run on CPU in a reasonable time
generator = pipeline(
    "text-generation",
    model="gpt2",
    pad_token_id=50256   # suppress padding warning
)

print("✅ GPT-2 model loaded!")

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ GPT-2 model loaded!


In [6]:
# --- Experiment 3: Basic text generation ---
prompt = "Artificial intelligence will change the world by"

output = generator(
    prompt,
    max_new_tokens=60,     # how many new tokens to generate
    num_return_sequences=1 # generate 1 completion
)

print("📝 Prompt:", prompt)
print("\n🤖 Generated:")
print(output[0]['generated_text'])

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📝 Prompt: Artificial intelligence will change the world by

🤖 Generated:
Artificial intelligence will change the world by 2050. It will have a profound impact on the way we live. It will change our lives. It will transform our lives.

So how will this affect us? We'll have to determine how to best use AI to better understand how we live and work. As we learn more about AI


---
##  Part 4: Effect of Temperature on Output

**Temperature** is a crucial hyperparameter that controls the *creativity* vs *predictability* of outputs.

- `temperature → 0`: Model almost always picks the most probable next token → **repetitive, safe**
- `temperature = 1`: Samples from the true distribution → **natural**
- `temperature > 1`: Increases randomness → **creative but potentially incoherent**

In [7]:
# --- Experiment 4: Temperature comparison ---
prompt = "The future of machine learning is"
temperatures = [0.1, 0.7, 1.5]

for temp in temperatures:
    result = generator(
        prompt,
        max_new_tokens=40,
        temperature=temp,
        do_sample=True,      # must be True to use temperature
        num_return_sequences=1
    )
    print(f"\n🌡️  Temperature = {temp}")
    print("-" * 50)
    print(result[0]['generated_text'])

print("\n📌 Notice how higher temperature = more varied, unpredictable text.")

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_return_sequences', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🌡️  Temperature = 0.1
--------------------------------------------------
The future of machine learning is in the hands of the AI community.

The future of machine learning is in the hands of the AI community.

The future of machine learning is in the hands of the AI community.


Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🌡️  Temperature = 0.7
--------------------------------------------------
The future of machine learning is bright and exciting. It may ultimately be a good thing for machines to learn, but it's not the only thing that makes them good at it.

The future of machine learning is bright and

🌡️  Temperature = 1.5
--------------------------------------------------
The future of machine learning is unknown."

Machine-to-human interfaces that solve data acquisition and control problems and enable the sharing of information with other people are a possibility that AI experts are in awe of right now, and

📌 Notice how higher temperature = more varied, unpredictable text.


---
##  Part 5: Prompt Engineering Techniques

**Prompt Engineering** is the practice of crafting effective inputs to guide model outputs.  
It's a core skill for working with generative AI systems in 2024–2026.

### Technique 1: Zero-Shot Prompting
Ask the model directly without any examples.

In [8]:
# --- Experiment 5: Zero-Shot vs Few-Shot ---

# ZERO-SHOT: No examples — just ask
zero_shot_prompt = "Translate English to French: 'Good morning, how are you?'"

zero_shot_result = generator(
    zero_shot_prompt,
    max_new_tokens=30,
    do_sample=False   # greedy decoding for deterministic output
)

print("🔵 ZERO-SHOT PROMPT:")
print(zero_shot_prompt)
print("\n🤖 Output:")
print(zero_shot_result[0]['generated_text'])

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔵 ZERO-SHOT PROMPT:
Translate English to French: 'Good morning, how are you?'

🤖 Output:
Translate English to French: 'Good morning, how are you?'

'Good morning, how are you?' 'Good morning, how are you?' 'Good morning, how are you?' 'Good morning,


In [9]:
# FEW-SHOT: Give examples first, then ask
few_shot_prompt = """Translate English to French:
English: Hello → French: Bonjour
English: Thank you → French: Merci
English: Good night → French: Bonne nuit
English: Good morning, how are you? → French:"""

few_shot_result = generator(
    few_shot_prompt,
    max_new_tokens=15,
    do_sample=False
)

print("🟢 FEW-SHOT PROMPT:")
print(few_shot_prompt)
print("\n🤖 Output:")
print(few_shot_result[0]['generated_text'])

print("\n📌 Few-shot prompting gives the model a pattern to follow → better results.")

Both `max_new_tokens` (=15) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🟢 FEW-SHOT PROMPT:
Translate English to French:
English: Hello → French: Bonjour
English: Thank you → French: Merci
English: Good night → French: Bonne nuit
English: Good morning, how are you? → French:

🤖 Output:
Translate English to French:
English: Hello → French: Bonjour
English: Thank you → French: Merci
English: Good night → French: Bonne nuit
English: Good morning, how are you? → French: Bonne nuit
English: Good morning, how are you? →

📌 Few-shot prompting gives the model a pattern to follow → better results.


### Technique 2: Role Prompting
Tell the model *who it is* — this shapes tone, style, and domain expertise.

In [10]:
# --- Experiment 6: Role prompting ---
# Without role
plain_prompt = "Explain what a neural network is."

# With role assigned
role_prompt = """You are a friendly teacher explaining to a 10-year-old student.
Explain what a neural network is in simple words:"""

for label, prompt in [("WITHOUT role", plain_prompt), ("WITH role", role_prompt)]:
    result = generator(prompt, max_new_tokens=60, do_sample=True, temperature=0.7)
    print(f"\n{'='*55}")
    print(f"📌 {label}")
    print(f"Prompt: {prompt}")
    print(f"\n🤖 Output:\n{result[0]['generated_text']}")

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📌 WITHOUT role
Prompt: Explain what a neural network is.

🤖 Output:
Explain what a neural network is. Imagine you want to know what the most common words in a sentence are. You can see that if you want to know what a word like "the" means or "the", then you need to determine how many words the word means. You can also see that if you want to know what a

📌 WITH role
Prompt: You are a friendly teacher explaining to a 10-year-old student.
Explain what a neural network is in simple words:

🤖 Output:
You are a friendly teacher explaining to a 10-year-old student.
Explain what a neural network is in simple words: a neural network is a "learning network" that acts as a means of learning. It's the most basic thing that humans do and it's a great way to learn. But even if you're not a computer scientist, you're still learning to use it and it's a pretty good way to


### Technique 3: Chain-of-Thought (CoT) Prompting
Ask the model to *think step by step* before giving the final answer. Improves logical reasoning.

In [11]:
# --- Experiment 7: Chain-of-Thought ---

# Direct answer prompt
direct_prompt = "If I have 5 apples and give away 2, then buy 3 more, how many do I have?"

# Chain-of-thought prompt
cot_prompt = """If I have 5 apples and give away 2, then buy 3 more, how many do I have?
Let's think step by step:
Step 1:"""

for label, p in [("DIRECT", direct_prompt), ("CHAIN-OF-THOUGHT", cot_prompt)]:
    r = generator(p, max_new_tokens=60, do_sample=False)
    print(f"\n{'='*55}")
    print(f"🔷 {label} PROMPTING")
    print(r[0]['generated_text'])

Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔷 DIRECT PROMPTING
If I have 5 apples and give away 2, then buy 3 more, how many do I have?

If I have 5 apples and give away 2, then buy 3 more, how many do I have?

If I have 5 apples and give away 2, then buy 3 more, how many do I have?

If I have 5 apples and give away 2, then buy

🔷 CHAIN-OF-THOUGHT PROMPTING
If I have 5 apples and give away 2, then buy 3 more, how many do I have?
Let's think step by step:
Step 1: Choose a new apple.
Step 2: Pick a new apple.
Step 3: Pick a new apple.
Step 4: Pick a new apple.
Step 5: Pick a new apple.
Step 6: Pick a new apple.
Step 7: Pick a new apple.



---
##  Part 6: Sentiment Analysis — GenAI for Classification

Modern LLMs can perform classification tasks *without explicit training* through prompting.  
Here, we use a dedicated sentiment analysis pipeline as a practical GenAI application.

In [12]:
# Load a sentiment analysis pipeline (uses DistilBERT under the hood)
sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Sample reviews to analyze
reviews = [
    "This course is absolutely amazing! I learned so much.",
    "The lab sessions are boring and not helpful at all.",
    "It was okay, nothing special but not bad either.",
    "Generative AI is the future and I'm excited to learn it!",
    "The exam was extremely difficult, I don't think I passed."
]

print(f"{'Review':<55} | {'Label':<10} | Score")
print("-" * 85)

results = sentiment_pipe(reviews)
for review, result in zip(reviews, results):
    print(f"{review[:52]:<55} | {result['label']:<10} | {result['score']:.4f}")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Review                                                  | Label      | Score
-------------------------------------------------------------------------------------
This course is absolutely amazing! I learned so much    | POSITIVE   | 0.9999
The lab sessions are boring and not helpful at all.     | NEGATIVE   | 0.9998
It was okay, nothing special but not bad either.        | POSITIVE   | 0.9891
Generative AI is the future and I'm excited to learn    | POSITIVE   | 0.9998
The exam was extremely difficult, I don't think I pa    | NEGATIVE   | 0.9986


---
##  Part 7: Text Summarization — GenAI Reducing Information

Summarization is one of the most practical applications of generative models.  
We use a sequence-to-sequence (seq2seq) model that maps long text → short summary.

In [15]:
!pip install sumy -q

from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer

long_text = """
Machine learning is a subfield of artificial intelligence that gives computers the ability 
to learn from data without being explicitly programmed. It has transformed industries 
ranging from healthcare and finance to transportation and entertainment. Modern machine 
learning techniques like deep learning have achieved superhuman performance on tasks such 
as image recognition, speech understanding, and strategic game playing. With the rise of 
large language models, the boundaries of what AI systems can do have expanded dramatically. 
These models, trained on billions of parameters and vast amounts of text, can generate 
coherent essays, write functional code, translate languages, and even engage in nuanced 
reasoning — capabilities that were science fiction just a decade ago. As of 2025, 
generative AI has become a mainstream tool used by students, developers, researchers, 
and creative professionals worldwide.
"""

# Parse and summarize — no model download needed
parser     = PlaintextParser.from_string(long_text, Tokenizer("english"))
summarizer = LsaSummarizer()
summary    = summarizer(parser.document, sentences_count=2)  # extract 2 key sentences

summary_text = " ".join(str(s) for s in summary)

print("📄 Original Text (word count):", len(long_text.split()))
print("\n📝 Summary (Extractive - LSA):")
print(summary_text)
print("\n📄 Summary (word count):", len(summary_text.split()))

📄 Original Text (word count): 131

📝 Summary (Extractive - LSA):
Machine learning is a subfield of artificial intelligence that gives computers the ability to learn from data without being explicitly programmed. As of 2025, generative AI has become a mainstream tool used by students, developers, researchers, and creative professionals worldwide.

📄 Summary (word count): 40


---
##  Part 8: Prompt Design Exercise

This section lets you practice designing prompts systematically.  
We compare **bad vs good** prompt design side by side.

In [16]:
# --- Experiment 8: Bad prompt vs Good prompt ---

prompts = {
    "❌ Vague Prompt": "Write something about climate.",
    
    "✅ Specific Prompt": (
        "Write a 3-sentence explanation of climate change suitable "
        "for a high school student. Focus on causes and one solution."
    ),
    
    "✅ Structured Prompt": (
        "Topic: Climate Change\n"
        "Audience: High school student\n"
        "Format: 3 bullet points\n"
        "Task: Explain the main causes of climate change.\n"
        "Response:"
    )
}

for label, prompt in prompts.items():
    result = generator(prompt, max_new_tokens=80, do_sample=True, temperature=0.7)
    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"Prompt: {prompt[:100]}..." if len(prompt) > 100 else f"Prompt: {prompt}")
    print(f"\n🤖 Output:\n{result[0]['generated_text']}")

Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



❌ Vague Prompt
Prompt: Write something about climate.

🤖 Output:
Write something about climate.

Climate is a problem. It is a problem that is too complex for science to solve. It is a problem that is much more complicated than it is for human beings. It is a problem that is impossible to solve without some kind of "fix" — a policy that would prevent people from doing something about it, or simply to change it.

We're not talking about a problem


Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✅ Specific Prompt
Prompt: Write a 3-sentence explanation of climate change suitable for a high school student. Focus on causes...

🤖 Output:
Write a 3-sentence explanation of climate change suitable for a high school student. Focus on causes and one solution.

What you should know

A great way to learn about climate change is by using the following techniques:

Make the story clear and clear.

If the situation is obvious, make it clear.

Keep a map of the climate change, so that you can focus on the important issues.

Make sure to make sure that the information is accurate.

Get

✅ Structured Prompt
Prompt: Topic: Climate Change
Audience: High school student
Format: 3 bullet points
Task: Explain the main c...

🤖 Output:
Topic: Climate Change
Audience: High school student
Format: 3 bullet points
Task: Explain the main causes of climate change.
Response:
1. The main cause is the impact of extreme weather events on human life on earth.
2. There is no simple answer.
3. If there is no simp